# Transfer Learning USAD — Detección de Anomalías de Temperatura SIATA
## Estación 203 (UNAL) — Plan B: Two-Stage Masking Training

**Estrategia:** Opción 3 — Transfer entre Stage 1 (datos limpios) y Stage 2 (masking Plan B)

**Restricciones:**
- Normalización con media y desviación estándar (sin MinMaxScaler)
- Sin `fillna()` ni `dropna()` — NA gestionados por masking
- Métricas: accuracy, precision, recall, f1, confusion_matrix
- Principios SOLID en la arquitectura del código

## 1. Setup — Clonar Repositorio desde GitHub

In [ ]:
import os

GITHUB_USER = "tu-usuario"              # <-- Cambiar por tu usuario GitHub
REPO_NAME   = "data-science-monograph" # <-- Cambiar por el nombre del repo
BRANCH      = "feature/transfer-learning-plan-b"

REPO_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

print(f"Repositorio disponible en: {REPO_DIR}")

In [ ]:
import sys

USAD_DIR = f"{REPO_DIR}/modelos/usad"
SRC_DIR  = f"{USAD_DIR}/src"

sys.path.insert(0, USAD_DIR)
sys.path.insert(0, SRC_DIR)

!pip install -q torch scikit-learn pandas numpy matplotlib seaborn

print("Paths listos.")

## 2. Imports y Configuración

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

# Módulos SOLID
from data_loader import SIATADataLoader
from normalizer import ZScoreNormalizer
from windowing import SlidingWindowBuilder
from trainer import Stage1Trainer, Stage2Trainer
from evaluator import AnomalyEvaluator
from reporter import MarkdownReporter

# Modelo USAD
from usad import UsadModel
from utils import get_default_device

torch.manual_seed(42)
np.random.seed(42)

DEVICE = get_default_device()
print(f"Dispositivo: {DEVICE}")

In [ ]:
WINDOW_SIZE = 60
W_SIZE      = WINDOW_SIZE      # 1 feature: temperatura
Z_SIZE      = 30               # espacio latente
BATCH_SIZE  = 512
EPOCHS_S1   = 50
EPOCHS_S2   = 50
ALPHA       = 0.5
BETA        = 0.5

DATA_PATH   = f"{USAD_DIR}/data/plan_b/203.csv"
OUTPUT_DIR  = f"{USAD_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = dict(
    window_size=WINDOW_SIZE, w_size=W_SIZE, z_size=Z_SIZE,
    batch_size=BATCH_SIZE, epochs_stage1=EPOCHS_S1, epochs_stage2=EPOCHS_S2,
    alpha=ALPHA, beta=BETA, device=str(DEVICE)
)

## 3. Carga de Datos

In [ ]:
# S — SIATADataLoader: única responsabilidad = cargar y partir datos
loader = SIATADataLoader(DATA_PATH)

df_train = loader.get_split("E")
df_val   = loader.get_split("V")
df_test  = loader.get_split("T")

print(f"Split E: {len(df_train):,} filas")
print(f"Split V: {len(df_val):,} filas")
print(f"Split T: {len(df_test):,} filas")
print(f"\nAnomalías reales en T (flag=1): {(df_test['flag'] == 1).sum():,}")
df_train.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=False)
for ax, (df, title, color) in zip(axes, [
    (df_train, "E — Entrenamiento", "steelblue"),
    (df_val,   "V — Validación",    "seagreen"),
    (df_test,  "T — Prueba",        "tomato"),
]):
    ax.plot(df["fecha_hora"], df["t"], color=color, linewidth=0.4, label="t")
    anoms = df[df["flag"] == 1]
    ax.scatter(anoms["fecha_hora"], anoms["t"], color="red", s=1, label="anomalía")
    ax.set_title(title)
    ax.set_ylabel("°C")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/serie_203.png", dpi=120)
plt.show()

## 4. Normalización Z-Score

In [ ]:
# S — ZScoreNormalizer: única responsabilidad = normalizar
# Fit SOLO sobre datos E limpios (t_mask != -2000)
normalizer = ZScoreNormalizer()
normalizer.fit(loader.get_train_normal_mask())

print(f"Media: {normalizer.mean_:.4f} °C")
print(f"Std:   {normalizer.std_:.4f} °C")

# Transformar t_mask (preserva centinelas -2000)
train_t_norm  = normalizer.transform(df_train["t_mask"]).values
val_t_norm    = normalizer.transform(df_val["t_mask"]).values
test_t_norm   = normalizer.transform(df_test["t"]).values  # T no tiene máscara

# t_mask crudas para el masking de pérdida
train_mask_raw = df_train["t_mask"].values.astype(np.float32)
val_mask_raw   = df_val["t_mask"].values.astype(np.float32)

print(f"\nCentinelas en train: {(train_t_norm == -2000.0).sum():,}")
print(f"Centinelas en val:   {(val_t_norm == -2000.0).sum():,}")

## 5. Ventanas Deslizantes

In [ ]:
# S — SlidingWindowBuilder: única responsabilidad = construir ventanas
wb = SlidingWindowBuilder(window_size=WINDOW_SIZE)

# Stage 1: solo datos E limpios
clean_mask = (train_t_norm != -2000.0) & ~np.isnan(train_t_norm)
s1_windows, _ = wb.build(train_t_norm[clean_mask])
print(f"Ventanas Stage 1: {s1_windows.shape}")

# Stage 2: todos los datos E + máscara
s2_win_data, s2_win_mask = wb.build(train_t_norm, train_mask_raw)
print(f"Ventanas Stage 2: {s2_win_data.shape}")

# Validación
val_win_data, val_win_mask = wb.build(val_t_norm, val_mask_raw)
print(f"Ventanas Val:     {val_win_data.shape}")

# Test
test_win_data, _ = wb.build(test_t_norm)
valid_test = ~np.isnan(test_win_data).any(axis=1)
test_win_data = test_win_data[valid_test]

# Labels de test: 1 si hay al menos una anomalía en la ventana
test_flags = df_test["flag"].values.astype(np.int32)
n_test = len(test_t_norm) - WINDOW_SIZE + 1
test_labels_all = np.array([
    1 if test_flags[i:i+WINDOW_SIZE].max() == 1 else 0
    for i in range(n_test)
])
test_labels = test_labels_all[valid_test]
print(f"Ventanas Test:    {test_win_data.shape} | Anomalías: {test_labels.sum():,}")

In [ ]:
# D — DataLoaders inyectados en los trainers
s1_loader  = DataLoader(TensorDataset(torch.FloatTensor(s1_windows)),
                        batch_size=BATCH_SIZE, shuffle=True)
s2_loader  = DataLoader(TensorDataset(torch.FloatTensor(s2_win_data),
                                       torch.FloatTensor(s2_win_mask)),
                        batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(val_win_data),
                                       torch.FloatTensor(val_win_mask)),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(test_win_data)),
                         batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches — S1:{len(s1_loader)}  S2:{len(s2_loader)}  Val:{len(val_loader)}  Test:{len(test_loader)}")

## 6. Stage 1 — Pre-entrenamiento (Datos Limpios)

In [ ]:
# D — modelo e optimizadores inyectados en el trainer
model = UsadModel(W_SIZE, Z_SIZE).to(DEVICE)

opt1_s1 = torch.optim.Adam(list(model.encoder.parameters()) + list(model.decoder1.parameters()))
opt2_s1 = torch.optim.Adam(list(model.encoder.parameters()) + list(model.decoder2.parameters()))

# O — Stage1Trainer extiende BaseTrainer sin modificarlo
s1_trainer = Stage1Trainer(model, opt1_s1, opt2_s1, DEVICE)

print("=== STAGE 1: Pre-entrenamiento ===")
history_s1 = s1_trainer.fit(s1_loader, val_loader, epochs=EPOCHS_S1)

torch.save({
    'encoder': model.encoder.state_dict(),
    'decoder1': model.decoder1.state_dict(),
    'decoder2': model.decoder2.state_dict(),
}, f"{OUTPUT_DIR}/model_stage1.pth")
print(f"\nPesos Stage 1 guardados.")

In [ ]:
ep = range(1, len(history_s1) + 1)
plt.figure(figsize=(10, 4))
plt.plot(ep, [h['val_loss1'] for h in history_s1], label='val_loss1')
plt.plot(ep, [h['val_loss2'] for h in history_s1], label='val_loss2')
plt.xlabel('Época'); plt.ylabel('Pérdida')
plt.title('Stage 1 — Convergencia')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_stage1.png", dpi=120)
plt.show()

## 7. Stage 2 — Fine-tuning con Masking Plan B (Transfer Learning)

In [ ]:
# Cargar pesos del Stage 1 → esto es el "transfer"
ckpt = torch.load(f"{OUTPUT_DIR}/model_stage1.pth", map_location=DEVICE)
model.encoder.load_state_dict(ckpt['encoder'])
model.decoder1.load_state_dict(ckpt['decoder1'])
model.decoder2.load_state_dict(ckpt['decoder2'])
print("Pesos Stage 1 cargados (transfer learning).")

opt1_s2 = torch.optim.Adam(list(model.encoder.parameters()) + list(model.decoder1.parameters()))
opt2_s2 = torch.optim.Adam(list(model.encoder.parameters()) + list(model.decoder2.parameters()))

# L — Stage2Trainer sustituye a Stage1Trainer (ambos derivan de BaseTrainer)
s2_trainer = Stage2Trainer(model, opt1_s2, opt2_s2, DEVICE)

print("\n=== STAGE 2: Fine-tuning con Masking Plan B ===")
history_s2 = s2_trainer.fit(s2_loader, val_loader, epochs=EPOCHS_S2)

torch.save({
    'encoder': model.encoder.state_dict(),
    'decoder1': model.decoder1.state_dict(),
    'decoder2': model.decoder2.state_dict(),
}, f"{OUTPUT_DIR}/model_stage2.pth")
print(f"\nPesos Stage 2 guardados.")

In [ ]:
ep = range(1, len(history_s2) + 1)
plt.figure(figsize=(10, 4))
plt.plot(ep, [h['val_loss1'] for h in history_s2], label='val_loss1')
plt.plot(ep, [h['val_loss2'] for h in history_s2], label='val_loss2')
plt.xlabel('Época'); plt.ylabel('Pérdida')
plt.title('Stage 2 — Convergencia (Fine-tuning con Masking)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_stage2.png", dpi=120)
plt.show()

## 8. Prueba del Modelo con Datos Reales

Se calcula el anomaly score sobre cada ventana del split T y se compara contra las etiquetas reales (`flag`). El threshold se determina maximizando F1 sobre validación.

In [ ]:
# I — AnomalyEvaluator: compute_scores, find_threshold y compute_metrics son independientes
evaluator = AnomalyEvaluator(device=DEVICE, alpha=ALPHA, beta=BETA)

# Scores sobre validación para encontrar el threshold
val_scores = evaluator.compute_scores(model, val_loader)

val_flags = df_val["flag"].values.astype(np.int32)
n_val_wins = len(val_win_data)
val_labels = np.array([
    1 if val_flags[i:i+WINDOW_SIZE].max() == 1 else 0
    for i in range(n_val_wins)
])

threshold = evaluator.find_threshold(val_scores, val_labels)
print(f"Threshold óptimo (max F1 en validación): {threshold:.6f}")

# Scores sobre TEST
test_scores = evaluator.compute_scores(model, test_loader)
print(f"\nScores test — min: {test_scores.min():.6f}  max: {test_scores.max():.6f}  media: {test_scores.mean():.6f}")
print(f"Ventanas predichas como anomalía: {(test_scores >= threshold).sum():,} / {len(test_scores):,}")

In [ ]:
# ============================================================
# MÉTRICAS FINALES SOBRE EL SPLIT T
# ============================================================
metrics = evaluator.compute_metrics(test_scores, test_labels, threshold)

print("=" * 50)
print("RESULTADOS — SPLIT T (Agosto-Septiembre 2023)")
print("=" * 50)
print(f"  Accuracy:   {metrics['accuracy']:.4f}")
print(f"  Precision:  {metrics['precision']:.4f}")
print(f"  Recall:     {metrics['recall']:.4f}")
print(f"  F1-Score:   {metrics['f1']:.4f}")
print(f"  Threshold:  {metrics['threshold']:.6f}")
print(f"  Anomalías predichas: {metrics['n_anomalies_predicted']:,}")
print(f"  Anomalías reales:    {metrics['n_anomalies_real']:,}")
print("\nConfusion Matrix:")
cm = metrics['confusion_matrix']
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomalía'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"Confusion Matrix — Split T\nF1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
# Distribución de scores: normal vs anomalía
plt.figure(figsize=(10, 4))
plt.hist([test_scores[test_labels == 0], test_scores[test_labels == 1]],
         bins=60, color=['#82E0AA', '#EC7063'], stacked=True,
         label=['Normal', 'Anomalía'])
plt.axvline(threshold, color='navy', linestyle='--', label=f'Threshold={threshold:.4f}')
plt.xlabel('Anomaly Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de Scores — Split T')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/score_distribution.png", dpi=120)
plt.show()

In [ ]:
# Línea de tiempo: temperatura real vs anomalías detectadas
n_valid_test = len(test_scores)
test_times = df_test["fecha_hora"].values[WINDOW_SIZE - 1 : WINDOW_SIZE - 1 + n_valid_test]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Panel 1: temperatura
ax1.plot(df_test["fecha_hora"], df_test["t"], color='steelblue', linewidth=0.4, label='Temperatura real')
anoms = df_test[df_test['flag'] == 1]
ax1.scatter(anoms['fecha_hora'], anoms['t'], color='red', s=1, label='Anomalía real (flag=1)')
ax1.set_ylabel('°C')
ax1.set_title('Temperatura Real — Estación 203 (UNAL)')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# Panel 2: anomaly scores
ax2.plot(test_times, test_scores, color='darkorange', linewidth=0.4, label='Anomaly Score')
ax2.axhline(threshold, color='navy', linestyle='--', linewidth=1, label=f'Threshold={threshold:.4f}')
detected = test_scores >= threshold
ax2.fill_between(test_times, 0, test_scores, where=detected, alpha=0.35, color='red', label='Detectado')
ax2.set_ylabel('Score')
ax2.set_xlabel('Fecha')
ax2.set_title('Anomaly Scores del Modelo')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/timeline_deteccion.png", dpi=120)
plt.show()

## 9. Exportar Reporte Final (.md)

In [ ]:
# S — MarkdownReporter: única responsabilidad = generar reporte
reporter = MarkdownReporter()
report_path = f"{OUTPUT_DIR}/reporte_final.md"

content = reporter.export(
    metrics=metrics,
    config=CONFIG,
    output_path=report_path,
    stage="Stage 2 — Transfer Learning con Masking Plan B",
)

print(f"Reporte exportado en: {report_path}")
print("\n" + content)

In [ ]:
# Descargar archivos desde Colab
from google.colab import files
files.download(report_path)
files.download(f"{OUTPUT_DIR}/model_stage2.pth")